# FAI Football CV — Fine-tune a player detector (v0.1)

The tracking pipeline uses a *generic person* detector (RF-DETR / COCO), which caught only ~half the players on night film. This notebook fixes that: you **label your own film once** and **fine-tune RF-DETR** into a football-player detector, then plug it back into the tracking notebook so it catches all 22 with a clean team split.

### The five steps
1. **Extract frames** from your film to label.
2. **Label** them in Roboflow (draw a box on every player).
3. **Download** the labeled dataset here.
4. **Fine-tune** RF-DETR on it (a few GPU hours).
5. **Use** the trained model in the tracking pipeline.

### Honest expectations
- Labeling is the work: plan on **~100–200 frames**, every player boxed (both teams, including piles). ~1–2 hours by hand. Garbage labels → garbage model.
- You need a **free Roboflow account** (for labeling + dataset hosting) and its **API key**.
- GPU runtime required. Training a few hundred frames for ~10–20 epochs on a Colab T4 is roughly 30–90 min.
- This makes a **player** detector — refs and sideline people are handled by labeling them out (skip them) or as a separate class you ignore.

## 0. Setup
Colab: **Runtime ▸ Change runtime type ▸ GPU**, then run the install.

In [ ]:
!pip -q install rfdetr roboflow supervision opencv-python-headless
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Extract frames to label
Pull evenly-spaced frames from your clip. Use varied plays/angles for a robust detector — re-run with different clips to build up your set. Upload a clip (Files panel) and set the path.

In [ ]:
import cv2, os

CLIP_PATH  = 'clip.mp4'   # your uploaded film clip
NUM_FRAMES = 120          # how many frames to pull for labeling

os.makedirs('frames', exist_ok=True)
cap = cv2.VideoCapture(CLIP_PATH)
assert cap.isOpened(), f'Could not open {CLIP_PATH}'
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
step = max(1, total // NUM_FRAMES)
i = saved = 0
while True:
    ok, fr = cap.read()
    if not ok: break
    if i % step == 0:
        cv2.imwrite(f'frames/frame_{i:05d}.jpg', fr); saved += 1
    i += 1
cap.release()
print(f'saved {saved} frames to ./frames')
!zip -qr frames.zip frames && echo 'zipped -> frames.zip  (download it from the Files panel to upload to Roboflow)'

## 2. Label in Roboflow (web app)
1. Create a free account at **roboflow.com** → **New Project** → type **Object Detection**.
2. Class list: one class, **`player`** (keep it simple). Optionally add `ref` if you'd rather label-and-ignore officials than skip them.
3. **Upload** `frames.zip` (drag it in), then open the annotation tool.
4. **Draw a tight box on every player** — both teams, linemen in a pile, everyone on the field. Skip refs, sideline, and crowd (or box them as `ref`).
5. When done: **Generate** a dataset version. Keep preprocessing simple (auto-orient + resize); light or no augmentation is fine for a first pass.
6. **Export** the version → format **COCO** → choose *show download code* → copy the `Roboflow(...)` snippet it gives you.

Tips that decide quality: label **consistently** (every player, tight boxes), and cover **varied looks** (different formations, both directions, near + far). More good frames beats more epochs.

## 3. Download your labeled dataset
Paste the snippet Roboflow gave you (it fills in workspace / project / version / key). Keep `.download("coco")` — RF-DETR trains on COCO format.

In [ ]:
from roboflow import Roboflow
# --- paste your export snippet, keeping COCO format ---
rf = Roboflow(api_key='YOUR_API_KEY')
project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
dataset = project.version(1).download('coco')
# -----------------------------------------------------
print('dataset at:', dataset.location)
!ls {dataset.location}

## 4. Fine-tune RF-DETR
Trains on your labeled frames. Start with ~15 epochs; watch the validation mAP. If you hit a GPU out-of-memory error, drop `batch_size` to 2 (and raise `grad_accum_steps` to keep the effective batch ~16).

In [ ]:
from rfdetr import RFDETRBase

EPOCHS = 15
model = RFDETRBase()
model.train(
    dataset_dir=dataset.location,
    epochs=EPOCHS,
    batch_size=4,
    grad_accum_steps=4,   # effective batch = batch_size * grad_accum_steps
    lr=1e-4,
    output_dir='rfdetr_football',
)
print('checkpoints:')
!ls -la rfdetr_football

## 5. Check it on a frame
Load the best checkpoint and run it on frame 0. You want the player count near the true ~22 and boxes on players it used to miss.

In [ ]:
import supervision as sv
from PIL import Image

BEST = 'rfdetr_football/checkpoint_best_total.pth'   # adjust to the file ls shows if named differently
ft = RFDETRBase(pretrain_weights=BEST)

cap = cv2.VideoCapture(CLIP_PATH); ok, fr = cap.read(); cap.release()
rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
det = ft.predict(Image.fromarray(rgb), threshold=0.4)
print('players detected on frame 0:', len(det))
print('class ids present:', set(int(c) for c in det.class_id))
sv.plot_image(sv.BoxAnnotator().annotate(rgb.copy(), det), size=(12, 7))

## 6. Use it in the tracking pipeline
Back in `football_tracking_pipeline.ipynb`, point the detector at your trained weights and set the class id to your `player` class (printed above):

```python
from rfdetr import RFDETRBase
model = RFDETRBase(pretrain_weights='rfdetr_football/checkpoint_best_total.pth')
CFG.person_class_id = 0   # set to the player class id printed in step 5
```

Keep the two notebooks on the **same Colab runtime** (or copy the checkpoint to Drive) so the weights are available. Then run the tracking notebook as before — detection should jump toward all 22, the team split gets cleaner, and the top-down radar starts to look like a real formation.

## 7. If results are weak
- **Missing players** → label more frames, especially the situations it misses (piles, far hash, motion blur).
- **Boxes on refs/sideline** → label those out (as `ref`) or crop tighter to the field.
- **mAP stuck low** → check label quality first; then try more epochs. Data quality beats hyperparameters.
- **OOM during training** → `batch_size=2, grad_accum_steps=8`.

Once the detector is solid, the same JSON export feeds the FAI Film Room importer — that's the bridge from this proof to the app.